# Spam Detection Using Naive Bayes

In [4]:
import numpy as np

## Collecting Data

In [5]:
spam = [
    "To use your credit, click the new WAP link in the next years txt message or click here", 
    "Thanks for your subscription to New Ringtone UK your new mobile will be charged £5/month Please confirm annoncement by replying", 
    "As a valued customer, I am pleased to advise you that following recent delivery waiting review of your Mob No. you are awarded with. Call us to review.", 
    "Please call our new customer service representative on", 
    "We are trying to contact you. Last weekends customer draw shows that you won a £1000 prize GUARANTEED. Calling years", 
]

In [6]:
# To check spam message
test_spam = ["Customer service annoncement. You have a New Years delivery waiting for you. click"]

In [7]:
non_spam = [
    "I don't think he goes to usf, he lives around here though", 
    "New car and house for my parents. i have only new job in hand", 
    "Great escape. I fancy the bridge but needs her lager. See you tomorrow", 
    "Tired. I haven't slept well the past few nights.",
    "Too late. I said i have the website. I didn't i have or dont have the slippers", 
    "I might come by tonight then if my class lets out early", 
    "Jos ask if u wana meet up?", 
    "That would be great. We'll be at the Guild. We can try meeting with the customer on Bristol road or somewhere"
    ]

In [8]:
# To check Non-spam
test_non_spam = ["That would be great. We'll be at the Guild. We can try meeting with the customer on Bristol road or somewhere"]

## Basic Pre-Processing

In [1]:
!pip install gensim

  Using cached smart_open-7.6.1-py3-none-any.whl.metadata (25 kB)
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   - -------------------------------------- 1.0/24.4 MB 5.0 MB/s eta 0:00:05
   -- ------------------------------------- 1.6/24.4 MB 4.4 MB/s eta 0:00:06
   --- ------------------------------------ 2.1/24.4 MB 3.6 MB/s eta 0:00:07
   --- ------------------------------------ 2.4/24.4 MB 3.4 MB/s eta 0:00:07
   --- ------------------------------------ 2.4/24.4 MB 3.4 MB/s eta 0:00:07
   --- ------------------------------------ 2.4/24.4 MB 3.4 MB/s eta 0:00:07
   --- ------------------------------------ 2.4/24.4 MB 3.4 MB/s eta 0:00:07
   ---- ----------------------------------- 2.6/24.4 MB 1.6 MB/s eta 0:00:14
   ---- ----------------------------------- 2.6/24.4 MB 1.6 MB/s eta 0:00:14
   ---- ----------------------------------- 2.6/24.4 MB 1.6 MB/s eta 0:00:14
   ------ --------------

In [2]:
from gensim.parsing.preprocessing import remove_stopwords
from gensim.parsing.porter import PorterStemmer
from gensim.utils import tokenize

In [16]:
test_sentence = non_spam[4]
test_sentence = non_spam[5]
test_sentence = non_spam[1]

print(test_sentence)

remove_stops = remove_stopwords(test_sentence)
print(remove_stops)

# Stemming
p = PorterStemmer()
stemmed = p.stem(remove_stops)
print(stemmed)

# Means you take something large and split it into smaller pieces based on some rules.
tokens = tokenize(stemmed)
print(list(tokens))

New car and house for my parents. i have only new job in hand
New car house parents. new job hand
new car house parents. new job hand
['new', 'car', 'house', 'parents', 'new', 'job', 'hand']


## Create a dictionary of words

In [20]:
# To automate the above work
def tokenize_sentence(sentence):
    p = PorterStemmer()
    remove_stops = remove_stopwords(sentence)
    stemmed = p.stem(remove_stops)
    tokens = tokenize(stemmed)
    return list(tokens)

In [21]:
dictionary = set()
spams_tokenize = []
non_spams_tokenize = []

for sentence in spam:
    sentence_tokens = tokenize_sentence(sentence)
    spams_tokenize.append(sentence_tokens)
    dictionary = dictionary.union(sentence_tokens) # add sentence words to the dictionary

for sentence in non_spam:
    sentence_tokens = tokenize_sentence(sentence)
    non_spams_tokenize.append(sentence_tokens)
    dictionary = dictionary.union(sentence_tokens)

print(f"Tokenized Spam: , {spams_tokenize}\nTokenized Non-Spam: {non_spams_tokenize}\nDictionary: {dictionary}")

Tokenized Spam: , [['to', 'use', 'credit', 'click', 'new', 'wap', 'link', 'years', 'txt', 'message', 'click'], ['thanks', 'subscription', 'new', 'ringtone', 'uk', 'new', 'mobile', 'charged', 'month', 'please', 'confirm', 'annoncement', 'repli'], ['as', 'valued', 'customer', 'i', 'pleased', 'advise', 'following', 'recent', 'delivery', 'waiting', 'review', 'mob', 'no', 'awarded', 'with', 'call', 'review'], ['please', 'new', 'customer', 'service', 'repres'], ['we', 'trying', 'contact', 'you', 'last', 'weekends', 'customer', 'draw', 'shows', 'won', 'prize', 'guaranteed', 'calling', 'year']]
Tokenized Non-Spam: [['i', 'don', 't', 'think', 'goes', 'usf', 'l'], ['new', 'car', 'house', 'parents', 'new', 'job', 'hand'], ['great', 'escape', 'i', 'fancy', 'bridge', 'needs', 'lager', 'see', 'tomorrow'], ['tired', 'i', 'haven', 't', 'slept', 'past', 'nights'], ['too', 'late', 'i', 'said', 'website', 'i', 'didn', 't', 'dont', 'slipp'], ['i', 'come', 'tonight', 'class', 'lets', 'earli'], ['jos', 'ask

## Basic Stats

In [25]:
# These things do not depend on an individual word so let's calculate them separately once
total_word_count = len(dictionary)
total_spam_messages = len(spams_tokenize)
total_non_spam_messages = len(non_spams_tokenize)
total_all_message = len(spams_tokenize) + len(non_spams_tokenize)

print("Total Number of Words: ",total_word_count)

Total Number of Words:  101


In [28]:
# P of spam: does not depend on an individual word so let's calculate that separately once 
# P of spem mean probability of spam
p_spam = total_spam_messages / total_all_message
print("P of spam = ", p_spam)

P of spam =  0.38461538461538464


In [29]:
# Helper function to count accurances.

def count_word_in_messages(word, messages):
    total_count = 0
    for msg in messages:
        if word in msg:
            total_count += 1

    return total_count

## The Actual Probability Computation

In [31]:
f_prob = 1

for test_sentence in test_spam:
    test_sent = tokenize_sentence(test_sentence)
    print(test_sent)

    # let's run this for each word separately
    for word in test_sent:
        print("---------------\nRunning for word:", word)

        # Find P(w | spam)
        spam_count = count_word_in_messages(word, spams_tokenize)
        p_w_spam = spam_count / total_spam_messages
        print("P( w | spam ) :",p_w_spam)

        # Find P(W)
        w_count = count_word_in_messages(word, spams_tokenize)
        w_count += count_word_in_messages(word, non_spams_tokenize)
        p_w = w_count / total_all_message
        print("P( w )        :",p_w)

        # Find P(spam | w)
        p_spam_w = (p_w_spam * p_spam) / p_w
        print("P( spam )     :",p_spam)
        print("P( spam | w ) :",p_spam_w)
        print("")
        f_prob *= p_spam_w

    print("P( spam | all_words ):",f_prob)

['customer', 'service', 'annoncement', 'you', 'new', 'years', 'delivery', 'waiting', 'you', 'click']
---------------
Running for word: customer
P( w | spam ) : 0.6
P( w )        : 0.3076923076923077
P( spam )     : 0.38461538461538464
P( spam | w ) : 0.75

---------------
Running for word: service
P( w | spam ) : 0.2
P( w )        : 0.07692307692307693
P( spam )     : 0.38461538461538464
P( spam | w ) : 1.0

---------------
Running for word: annoncement
P( w | spam ) : 0.2
P( w )        : 0.07692307692307693
P( spam )     : 0.38461538461538464
P( spam | w ) : 1.0

---------------
Running for word: you
P( w | spam ) : 0.2
P( w )        : 0.07692307692307693
P( spam )     : 0.38461538461538464
P( spam | w ) : 1.0

---------------
Running for word: new
P( w | spam ) : 0.6
P( w )        : 0.3076923076923077
P( spam )     : 0.38461538461538464
P( spam | w ) : 0.75

---------------
Running for word: years
P( w | spam ) : 0.2
P( w )        : 0.07692307692307693
P( spam )     : 0.3846153846153